In [0]:
# ============================================================
# CELL 0 — Config
# ============================================================
catalog = dbutils.widgets.get("catalog") if "catalog" in [w.name for w in dbutils.widgets.getAll()] else "main"
schema  = dbutils.widgets.get("schema")  if "schema"  in [w.name for w in dbutils.widgets.getAll()] else "worldcup"
table   = dbutils.widgets.get("table")   if "table"   in [w.name for w in dbutils.widgets.getAll()] else "bronze_raw_fifa_wc_2026_teams"

download_path = "/tmp/fifa_wc_2026"
uc_volume_path = "/Volumes/" + catalog + "/" + schema + "/landing"
bronze_table_path = "/Volumes/" + catalog + "/" + schema + "/landing/delta/" + table
version_path      = uc_volume_path + "/last_known_version.txt"

In [0]:
# ============================================================
# CELL 1 — Install Kaggle if not already installed
# ============================================================
try:
    import kaggle
    print("Kaggle already installed — skipping pip install")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "kaggle"], check=True)
    dbutils.library.restartPython()
    print("Kaggle installed and Python runtime restarted")
    import kaggle

In [0]:
# ============================================================
# CELL 2 — Retrieve secrets from Databricks secret scope
# ============================================================
kaggle_username = dbutils.secrets.get(scope="application-secrets", key="kaggle_username")
kaggle_key      = dbutils.secrets.get(scope="application-secrets", key="kaggle_key")

In [0]:
# ============================================================
# CELL 3 — Write Kaggle credentials to ~/.kaggle/kaggle.json
# ============================================================
import json
import os

# Use /tmp instead of /root/.kaggle — Free Tier restricts /root writes
kaggle_dir = "/tmp/.kaggle"
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_creds = {"username": kaggle_username, "key": kaggle_key}

with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)

os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

# Tell the Kaggle library where to find the credentials file
os.environ["KAGGLE_CONFIG_DIR"] = kaggle_dir

print("Kaggle credentials written.")

In [0]:
# ============================================================
# CELL 4 — Check Kaggle dataset version, exit early if unchanged
# ============================================================

kaggle.api.authenticate()

datasets     = kaggle.api.dataset_list(search="swaptr/fifa-wc-2026-teams")
dataset_meta = next((d for d in datasets if d.ref == "swaptr/fifa-wc-2026-teams"), None)

if dataset_meta is None:
    raise Exception("Dataset swaptr/fifa-wc-2026-teams not found via Kaggle API")

current_version = str(dataset_meta.current_version_number)
print("Current version: " + current_version)

# Read last known version from UC Volume
try:
    last_version = dbutils.fs.head(version_path).strip()
    print("Last ingested version: " + last_version)
except:
    last_version = ""
    print("No prior version found — first run")

if current_version == last_version:
    print("Dataset unchanged since last ingest — skipping")
    dbutils.notebook.exit("SKIPPED - already on version " + current_version)
else:
    print("Dataset changed (" + last_version + " -> " + current_version + ") — proceeding with ingest")

In [0]:
# ============================================================
# CELL 5 — Download the dataset
# ============================================================

# Downloads to /tmp/fifa_wc_2026 on the driver node
os.makedirs(download_path, exist_ok=True)

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    dataset   = "swaptr/fifa-wc-2026-teams",
    path      = download_path,
    unzip     = True,
    quiet     = False
)

print(f"Files downloaded: {os.listdir(download_path)}")

In [0]:
# ============================================================
# CELL 6 — Copy raw files to Unity Catalog Volume
# ============================================================
import shutil

dbutils.fs.mkdirs(uc_volume_path)

for fname in os.listdir(download_path):
    src  = download_path + "/" + fname
    dest = uc_volume_path + "/" + fname
    shutil.copy(src, dest)
    print("Copied " + fname + " → " + dest)

In [0]:
# ============================================================
# CELL 7 — Read CSVs and write to Bronze Delta table
# ============================================================
from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mergeSchema", "true")
    .csv(uc_volume_path + "/*.csv")
)

df_bronze = (
    df_raw
    .withColumn("_ingested_at",    F.current_timestamp())
    .withColumn("_source_file",    F.col("_metadata.file_path"))
    .withColumn("_ingestion_date", F.to_date(F.current_timestamp()))
)

print("Row count: " + str(df_bronze.count()))
df_bronze.printSchema()

In [0]:
# ============================================================
# CELL 8 — Register table in Unity Catalog (3-level namespace)
# ============================================================

# Schema already exists from Cell 0 config, but ensure it's created
spark.sql("CREATE SCHEMA IF NOT EXISTS " + catalog + "." + schema)

full_table_name = catalog + "." + schema + "." + table

# Write directly as a managed UC table instead of using LOCATION
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql("SELECT * FROM " + full_table_name + " LIMIT 5").show(truncate=False)

In [0]:
# ============================================================
# CELL 9 — Persist current version number after successful ingest
# ============================================================
dbutils.fs.put(version_path, current_version, overwrite=True)
print("Saved version " + current_version + " to " + version_path)